# 02 – Tiền xử lý (Preprocessing)
Xem từng bước `TextCleaner` làm gì, rồi **đo tác động của từng bước** (mỗi lần bật thêm một bước).

In [ ]:
import sys
sys.path.insert(0, "..")  # để import được package spam_filter
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

from spam_filter.config import DataConfig
from spam_filter.data import load_dataset, split_data

cfg = DataConfig()
df = load_dataset(cfg)
X_train, X_test, y_train, y_test = split_data(df, cfg)

In [ ]:
from spam_filter.pipeline import ExperimentSpec
from spam_filter.runner import run_specs

SHOW = ["id", "cv_f1_train", "precision", "recall", "f1"]

def compare(specs):
    """Chạy các spec, trả về bảng gọn (CV trên train + test). Chọn theo cv_f1_train."""
    res, _ = run_specs(specs, X_train, y_train, X_test, y_test, cfg.cv_folds)
    return res[SHOW].round(4).set_index("id")

## 1. Trước/sau tiền xử lý

In [ ]:
from spam_filter.preprocessing import TextCleaner
sample = df[df.label == 1].text.head(3).tolist() + df[df.label == 0].text.head(2).tolist()
pd.DataFrame({
    "gốc": sample,
    "lowercase": TextCleaner().transform(sample),
    "token hóa URL/tiền/số + bỏ dấu câu": TextCleaner(replace_url=True, replace_money=True, replace_number=True, remove_punct=True).transform(sample),
    "bỏ stopword": TextCleaner(remove_punct=True, remove_stopwords=True).transform(sample),
})

## 2. Ablation – tác động của từng bước
Cố định: TF-IDF (1,2)-gram, `min_df=2`, MultinomialNB `alpha=0.1`. Chỉ thay đổi cấu hình `TextCleaner`.

In [ ]:
FIXED = dict(model="nb", model_params={"alpha": 0.1},
             vectorizer={"kind": "tfidf", "ngram_range": (1, 2), "min_df": 2})
steps = {
    "chỉ lowercase": {},
    "+ thay URL": {"replace_url": True},
    "+ thay tiền": {"replace_money": True},
    "+ thay số": {"replace_number": True},
    "+ bỏ dấu câu": {"remove_punct": True},
    "+ bỏ stopword": {"remove_stopwords": True},
    "URL+tiền+số+dấu câu": {"replace_url": True, "replace_money": True, "replace_number": True, "remove_punct": True},
}
compare([ExperimentSpec(id=k, description=k, cleaner=v, **FIXED) for k, v in steps.items()])

## Nhận xét (nhóm điền)
Bước nào giúp ích / gây hại? Vì sao (liên hệ EDA: số, URL, tiền là tín hiệu spam)? Chênh lệch nhỏ có thể chỉ là nhiễu vì tập test có ít spam; đối chiếu CV F1 trên train với metric held-out test.